# 00 — Data Processing & Exploratory Data Analysis

**Inputs:**
- `medqa-data/questions/US/{train,dev,test}.jsonl` — MedQA US, 5-option variant
- `medqa-data/questions/US/4_options/phrases_no_exclude_{train,dev,test}.jsonl` — 4-option variant with `metamap_phrases`
- `medqa-data/textbooks/en/*.txt` — 18 English medical textbooks (the retrieval knowledge base)

**Outputs (saved to `data/processed/`):**
- `medqa_5opt.parquet`, `medqa_4opt.parquet` — cleaned question dataframes
- `textbook_stats.parquet` — per-book corpus statistics
- `eda_summary.json` — headline numbers reused by later notebooks

**What this notebook covers:**
1. Load both MedQA variants and the 18 textbooks
2. Schema validation, deduplication, missing-value checks
3. Question-level EDA: length distribution, USMLE step mix, answer-letter balance, metamap phrase counts, long-vignette flag
4. Textbook-corpus EDA: per-book word counts, dominance of `InternalMed_Harrison` (a known retrieval-bias source — see `docs/golden-data/analysis.md` §7)
5. Save processed artifacts that subsequent experiment notebooks (`01_base_llm.ipynb` onward) can load instantly


## 1. Setup

In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve repo root: notebook may be opened from either repo root or notebooks/
cwd = Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

MEDQA_DIR = REPO_ROOT / "medqa-data"
QUESTIONS_DIR = MEDQA_DIR / "questions" / "US"
TEXTBOOKS_DIR = MEDQA_DIR / "textbooks" / "en"
OUTPUT_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_colwidth", 120)

assert MEDQA_DIR.exists(), f"MedQA dir missing: {MEDQA_DIR}"
assert QUESTIONS_DIR.exists(), f"Questions dir missing: {QUESTIONS_DIR}"
assert TEXTBOOKS_DIR.exists(), f"Textbooks dir missing: {TEXTBOOKS_DIR}"

print(f"Repo root:    {REPO_ROOT}")
print(f"MedQA dir:    {MEDQA_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")


## 2. Load MedQA question splits

Two parallel datasets exist: the original **5-option** version and a **4-option** variant where one distractor has been removed and `metamap_phrases` (pre-extracted clinical concepts) are attached. Both are loaded — most experiments will use the 4-option variant because it carries the metamap field and matches what the proposal uses, but accuracy reporting on the 5-option variant is preserved as a sanity check.


In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

# 5-option (canonical MedQA)
splits_5opt = {}
for split in ["train", "dev", "test"]:
    df = load_jsonl(QUESTIONS_DIR / f"{split}.jsonl")
    df["split"] = split
    splits_5opt[split] = df
    print(f"5-option {split}: {len(df):>6,} questions")

medqa_5opt = pd.concat(splits_5opt.values(), ignore_index=True)
print(f"5-option TOTAL:  {len(medqa_5opt):>6,} questions")
print(f"Columns: {list(medqa_5opt.columns)}")
medqa_5opt.head(2)


In [ ]:
# 4-option (with metamap_phrases)
splits_4opt = {}
for split in ["train", "dev", "test"]:
    df = load_jsonl(QUESTIONS_DIR / "4_options" / f"phrases_no_exclude_{split}.jsonl")
    df["split"] = split
    splits_4opt[split] = df
    print(f"4-option {split}: {len(df):>6,} questions")

medqa_4opt = pd.concat(splits_4opt.values(), ignore_index=True)
print(f"4-option TOTAL:  {len(medqa_4opt):>6,} questions")
print(f"Columns: {list(medqa_4opt.columns)}")
print(f"\nNew columns vs 5-option: {set(medqa_4opt.columns) - set(medqa_5opt.columns)}")
medqa_4opt.head(1)


## 3. Schema validation & quality checks

In [ ]:
def quality_report(df: pd.DataFrame, name: str) -> dict:
    print(f"=== {name} ({len(df):,} rows) ===")
    report = {"name": name, "rows": len(df)}
    for col in ["question", "answer", "answer_idx", "options", "meta_info"]:
        if col in df.columns:
            n_missing = df[col].isna().sum()
            print(f"  missing {col:<12}: {n_missing}")
            report[f"missing_{col}"] = int(n_missing)

    n_dup_q = df["question"].duplicated().sum()
    print(f"  duplicate questions  : {n_dup_q}")
    report["duplicate_questions"] = int(n_dup_q)

    n_options_per_q = df["options"].apply(len)
    print(f"  options/question      : min={n_options_per_q.min()}, "
          f"max={n_options_per_q.max()}, mode={n_options_per_q.mode().iloc[0]}")
    report["options_per_q_min"] = int(n_options_per_q.min())
    report["options_per_q_max"] = int(n_options_per_q.max())

    def matches(row):
        return row["options"].get(row["answer_idx"]) == row["answer"]

    n_mismatch = (~df.apply(matches, axis=1)).sum()
    print(f"  answer_idx ↔ answer  : {n_mismatch} mismatches")
    report["answer_idx_mismatches"] = int(n_mismatch)
    return report

report_5opt = quality_report(medqa_5opt, "MedQA 5-option")
print()
report_4opt = quality_report(medqa_4opt, "MedQA 4-option")


## 4. Question-level EDA

### 4.1 Question-text length distribution

The proposal and golden-data methodology use **>200 words** as the threshold for *long-vignette* clinical scenarios. We flag those and break length down by USMLE step.


In [ ]:
medqa_5opt["question_word_count"] = medqa_5opt["question"].str.split().str.len()
medqa_5opt["question_char_count"] = medqa_5opt["question"].str.len()
medqa_5opt["is_long_vignette"] = medqa_5opt["question_word_count"] > 200

print(medqa_5opt[["question_word_count", "question_char_count"]].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(medqa_5opt["question_word_count"], bins=50, ax=axes[0])
axes[0].axvline(200, color="red", linestyle="--", label="long-vignette threshold (200 words)")
axes[0].set_title("Question length distribution (words)")
axes[0].set_xlabel("words per question")
axes[0].legend()

sns.boxplot(data=medqa_5opt, x="split", y="question_word_count", ax=axes[1])
axes[1].set_title("Question length by split")
axes[1].set_ylabel("words per question")
plt.tight_layout()
plt.show()

n_long = int(medqa_5opt["is_long_vignette"].sum())
pct_long = medqa_5opt["is_long_vignette"].mean()
print(f"\nLong-vignette questions (>200 words): {n_long:,}  ({pct_long:.1%})")


### 4.2 Correct-answer letter balance — checks for label bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ans_5opt = medqa_5opt["answer_idx"].value_counts().sort_index()
ans_5opt.plot(kind="bar", ax=axes[0])
axes[0].set_title("5-option · correct-answer letter distribution")
axes[0].set_xlabel("answer_idx")
axes[0].set_ylabel("count")

ans_4opt = medqa_4opt["answer_idx"].value_counts().sort_index()
ans_4opt.plot(kind="bar", ax=axes[1], color="C1")
axes[1].set_title("4-option · correct-answer letter distribution")
axes[1].set_xlabel("answer_idx")
plt.tight_layout()
plt.show()

print("5-option (% per letter):")
print((ans_5opt / ans_5opt.sum() * 100).round(2).to_string())
print("\n4-option (% per letter):")
print((ans_4opt / ans_4opt.sum() * 100).round(2).to_string())


### 4.3 USMLE step mix (`meta_info`)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
meta = medqa_5opt["meta_info"].value_counts()
meta.plot(kind="bar", ax=ax)
ax.set_title("USMLE step distribution")
ax.set_xlabel("meta_info")
ax.set_ylabel("question count")
plt.tight_layout()
plt.show()

print("Step distribution (5-option):")
print(meta.to_string())
print("\nStep × Split crosstab:")
print(pd.crosstab(medqa_5opt["split"], medqa_5opt["meta_info"]))
print("\nLong-vignette × Step crosstab:")
print(pd.crosstab(medqa_5opt["is_long_vignette"], medqa_5opt["meta_info"]))


### 4.4 Metamap phrases per question (4-option dataset only)

In [ ]:
medqa_4opt["n_metamap_phrases"] = medqa_4opt["metamap_phrases"].str.len()
print(medqa_4opt["n_metamap_phrases"].describe().round(2))

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(medqa_4opt["n_metamap_phrases"], bins=40, ax=ax)
ax.set_title("Number of metamap phrases per question (4-option dataset)")
ax.set_xlabel("phrases per question")
plt.tight_layout()
plt.show()


### 4.5 Option-text length

In [ ]:
def option_word_lengths(opts):
    return [len(v.split()) for v in opts.values()]

medqa_5opt["option_word_lengths"] = medqa_5opt["options"].apply(option_word_lengths)
medqa_5opt["avg_option_word_count"] = medqa_5opt["option_word_lengths"].apply(np.mean)
medqa_5opt["max_option_word_count"] = medqa_5opt["option_word_lengths"].apply(max)

print(medqa_5opt[["avg_option_word_count", "max_option_word_count"]].describe().round(2))

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(medqa_5opt["avg_option_word_count"], bins=50, ax=ax)
ax.set_title("Average option length per question (words)")
ax.set_xlabel("avg words per option")
plt.tight_layout()
plt.show()

print(f"\nMean question length:        {medqa_5opt['question_word_count'].mean():.2f} words")
print(f"Mean avg-option length:      {medqa_5opt['avg_option_word_count'].mean():.2f} words")
print(f"Question:option length ratio ≈ "
      f"{medqa_5opt['question_word_count'].mean() / medqa_5opt['avg_option_word_count'].mean():.0f}×")


## 5. Textbook corpus EDA

The 18 English textbooks are the retrieval knowledge base for every RAG architecture. We profile their sizes and confirm the known **InternalMed_Harrison dominance** (~25% of corpus by word count — the source of the retrieval bias documented in `docs/golden-data/analysis.md` §7).


In [ ]:
textbook_files = sorted(TEXTBOOKS_DIR.glob("*.txt"))
print(f"Found {len(textbook_files)} English textbooks:\n")

books = {}
book_stats = []
for path in textbook_files:
    text = path.read_text(encoding="utf-8", errors="ignore")
    books[path.stem] = text
    book_stats.append({
        "book": path.stem,
        "file_size_mb": round(path.stat().st_size / 1024**2, 2),
        "char_count": len(text),
        "word_count": len(text.split()),
        "line_count": text.count("\n"),
    })

book_stats_df = (
    pd.DataFrame(book_stats)
      .sort_values("word_count", ascending=False)
      .reset_index(drop=True)
)
total_words = int(book_stats_df["word_count"].sum())
book_stats_df["share_pct"] = (book_stats_df["word_count"] / total_words * 100).round(2)

print(f"Total corpus: {total_words:,} words across {len(books)} books "
      f"({book_stats_df['file_size_mb'].sum():.1f} MB)")
print()
print(book_stats_df[["book", "word_count", "share_pct", "file_size_mb"]].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
plot_data = book_stats_df.sort_values("word_count")
ax.barh(plot_data["book"], plot_data["word_count"])
ax.set_xlabel("word count")
ax.set_title("Per-textbook word count — corpus size dominated by Harrison's")
plt.tight_layout()
plt.show()

top3 = book_stats_df.head(3)
print("Top 3 books account for "
      f"{top3['share_pct'].sum():.1f}% of corpus by word count:")
for _, row in top3.iterrows():
    print(f"  {row['book']:<30s}  {row['share_pct']:>5.2f}%  ({row['word_count']:>10,} words)")


In [ ]:
# Sample passages from a few representative books to eyeball formatting & quality
sample_books = ["First_Aid_Step1", "Pharmacology_Katzung", "InternalMed_Harrison"]
for name in sample_books:
    if name in books:
        snippet = books[name][:600].replace("\n", " ")
        print(f"--- {name} (first 600 chars) ---")
        print(snippet)
        print()


## 6. Cross-dataset signal — top metamap phrases

The `metamap_phrases` field gives a coarse view of what clinical concepts dominate the question pool. This is useful for sanity-checking later that the textbook corpus actually covers the topics being asked about.


In [ ]:
all_phrases = []
for phrases in medqa_4opt["metamap_phrases"]:
    all_phrases.extend(p.lower() for p in phrases)

phrase_counts = Counter(all_phrases)
print(f"Unique metamap phrases: {len(phrase_counts):,}")
print(f"Total phrase tokens:    {sum(phrase_counts.values()):,}")
print(f"\nTop 30 phrases by frequency:\n")
for phrase, count in phrase_counts.most_common(30):
    print(f"  {count:>6,}  {phrase}")


## 7. Save processed artifacts to `data/processed/`

In [ ]:
# 5-option dataset
out_5opt = medqa_5opt.copy()
out_5opt["options_json"] = out_5opt["options"].apply(json.dumps)
out_5opt["option_word_lengths_json"] = out_5opt["option_word_lengths"].apply(json.dumps)
out_5opt = out_5opt.drop(columns=["options", "option_word_lengths"])
out_5opt.to_parquet(OUTPUT_DIR / "medqa_5opt.parquet", index=False)
print(f"Saved: {OUTPUT_DIR / 'medqa_5opt.parquet'}  ({len(out_5opt):,} rows, {len(out_5opt.columns)} cols)")

# 4-option dataset
out_4opt = medqa_4opt.copy()
out_4opt["options_json"] = out_4opt["options"].apply(json.dumps)
out_4opt["metamap_phrases_json"] = out_4opt["metamap_phrases"].apply(json.dumps)
out_4opt = out_4opt.drop(columns=["options", "metamap_phrases"])
out_4opt.to_parquet(OUTPUT_DIR / "medqa_4opt.parquet", index=False)
print(f"Saved: {OUTPUT_DIR / 'medqa_4opt.parquet'}  ({len(out_4opt):,} rows, {len(out_4opt.columns)} cols)")

# Textbook stats
book_stats_df.to_parquet(OUTPUT_DIR / "textbook_stats.parquet", index=False)
print(f"Saved: {OUTPUT_DIR / 'textbook_stats.parquet'}  ({len(book_stats_df):,} books)")

# Headline EDA summary as JSON for easy consumption by later notebooks
summary = {
    "medqa_5opt": {
        "rows": len(medqa_5opt),
        "by_split": medqa_5opt["split"].value_counts().to_dict(),
        "by_meta_info": medqa_5opt["meta_info"].value_counts().to_dict(),
        "long_vignette_count": int(medqa_5opt["is_long_vignette"].sum()),
        "long_vignette_pct": round(float(medqa_5opt["is_long_vignette"].mean() * 100), 2),
        "answer_letter_pct": (medqa_5opt["answer_idx"].value_counts(normalize=True) * 100).round(2).to_dict(),
        "question_word_count_mean": round(float(medqa_5opt["question_word_count"].mean()), 2),
        "question_word_count_median": int(medqa_5opt["question_word_count"].median()),
    },
    "medqa_4opt": {
        "rows": len(medqa_4opt),
        "answer_letter_pct": (medqa_4opt["answer_idx"].value_counts(normalize=True) * 100).round(2).to_dict(),
        "n_metamap_phrases_mean": round(float(medqa_4opt["n_metamap_phrases"].mean()), 2),
        "n_metamap_phrases_median": int(medqa_4opt["n_metamap_phrases"].median()),
    },
    "textbooks": {
        "n_books": len(book_stats_df),
        "total_words": total_words,
        "top_3_books_share_pct": round(float(book_stats_df.head(3)["share_pct"].sum()), 2),
        "harrison_share_pct": round(
            float(book_stats_df.loc[book_stats_df["book"] == "InternalMed_Harrison", "share_pct"].iloc[0]), 2
        ) if "InternalMed_Harrison" in book_stats_df["book"].values else None,
    },
    "quality_report": {
        "medqa_5opt": report_5opt,
        "medqa_4opt": report_4opt,
    },
}

(OUTPUT_DIR / "eda_summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUTPUT_DIR / 'eda_summary.json'}")
print()
print(json.dumps(summary, indent=2))


## 8. Findings & next steps

**Headline numbers** (cross-check against `eda_summary.json`):

- **MedQA US (5-option):** 12,723 questions — train 10,178 / dev 1,272 / test 1,273. Matches the proposal's count exactly.
- **MedQA US (4-option):** same row count, with `metamap_phrases` attached. Use this variant for retrieval-side experiments because the phrases are a free clinical-concept signal.
- **Long-vignette rate** (>200 words) drives Multi-Hop RAG's expected advantage — track it.
- **InternalMed_Harrison dominates** the textbook corpus (~25% of words). Already documented in `docs/golden-data/analysis.md` §7 as a corpus-frequency bias source. Acknowledge in the discussion chapter; cancels out for *relative* architecture comparison.

**What's now ready for the next notebooks:**
- `data/processed/medqa_5opt.parquet`
- `data/processed/medqa_4opt.parquet`
- `data/processed/textbook_stats.parquet`
- `data/processed/eda_summary.json`

**Next notebook (`01_text_chunking_and_indexing.ipynb` — to build):**
1. Recursive 200-token / 20-token-overlap chunking of the 18 textbooks → `chunks.parquet`
2. Embedding generation with the chosen model → `embeddings.npy`
3. FAISS index + BM25 index build → `data/indices/`

Once those indices exist, every experiment notebook (`01_base_llm.ipynb` → `16_final_comparative_analysis.ipynb`) just *loads* them — no rebuild required per experiment.
